# Construcción de Zonas Agricolas

In [ ]:
# Proceso de selcción de los puntos para las zonas agricolas.

from config.paths import datapaths
from config.links import USOV_API
import geopandas as gpd
filenames = [file.split("/")[-1].replace(".zip", "") for file in USOV_API]
print(filenames)

map2005= datapaths['raw']['INEGI'] / "702825007022_s"/"shp"/"us250s3g"/"us250s3g.shp"
map2009= datapaths['raw']['INEGI'] / "702825007023_s"/"shape"/"us250s4g"/"us250s4g.shp"
map2013= datapaths['raw']['INEGI'] / "702825007024_s"/"conjunto_de_datos"/"CAPAS"/"usv250s5g"/"usv250s5g.shp"
map2016= datapaths['raw']['INEGI'] / "889463173359_s"/"conjunto_de_datos"/"usv250s6_union.shp"
map2017= datapaths['raw']['INEGI'] / "889463598459_s"/"conjunto_de_datos"/"usv250s6_union.shp"
map2018= datapaths['raw']['INEGI'] / "889463598459_s"/"conjunto_de_datos"/"usv250s6g.shp"
map2019= datapaths['raw']['INEGI'] / "889463842781_s"/"conjunto_de_datos"/"cdv_usuev250sVII_cnal.shp"

usov_maps = [map2005, map2009, map2013, map2016, map2017, map2018, map2019]

files = []
for us_map in usov_maps:
    print(us_map)
    df = gpd.read_file(us_map)
    files.append(df)
    print(df.shape)


# Edicion 2005

In [ ]:
df1 = files[0]
print(df1['AGECOSIS'].unique())
agro1 = df1[df1['AGECOSIS']=="AGRICOLA"]

agro1_points = agro1[['CODIGO','geometry']]
agro1_points['Centro'] = agro1_points.geometry.centroid
agro1_points

# Incorporamos Edicion 2009

In [ ]:
df2 = files[1]
agro2 = df2[df2['AGECOSIS']=="AGRICOLA"]

agro2_points = agro2[['CODIGO','geometry']]
agro2_points['Centro'] = agro2_points.geometry.centroid
agro2_points

In [ ]:
import pandas as pd

def add_points(base, new):
    distances = pd.DataFrame(index=base['Centro'], columns=new['Centro'])
    dims = distances.shape
    t = 0
    for p in distances.columns:
        if t % 1000==0:
            print(t / dims[1] * 100)
        distances[p] = base['Centro'].apply(lambda x: x.distance(p)).tolist()
        t+=1
    return distances[distances > 9_000]

pass1 = add_points(agro1_points, agro2_points)

In [ ]:
new_points = pass1[pass1 > 9_000].dropna(axis=1, how="any").columns

df_new = agro2_points[agro2_points['Centro'].isin(new_points)].reset_index(drop=True)
df_new

In [ ]:
agro12_points = pd.concat([agro1_points, df_new]).reset_index(drop=True)
agro12_points

In [ ]:
df3 = files[2]
print(df3['AGECOSIS'].unique())
agro3 = df3[df3['AGECOSIS']=="AGRÍCOLA"]

agro3_points = agro3[['CODIGO','geometry']]
agro3_points['Centro'] = agro3_points.geometry.centroid
agro3_points

In [ ]:
pass2 = add_points(agro12_points, agro3_points)
new_points = pass2[pass2 > 9_000].dropna(axis=1, how="any").columns


In [ ]:
df_new = agro3_points[agro3_points['Centro'].isin(new_points)]
agro13_points = pd.concat([agro12_points, df_new]).reset_index(drop=True)
agro13_points

In [ ]:
df4 = files[3]
agro4 = df4[df4['DESCRIPCIO'].isin([c for c in df4['DESCRIPCIO'] if c.startswith("AGR")])]
agro4['Centro'] = agro4.geometry.centroid
agro4_points = agro4[['OBJECTID', 'Centro']].reset_index(drop=True)
agro4_points


In [ ]:
pass3 = add_points(agro13_points, agro4_points)


In [ ]:
df_new = agro4_points[agro4_points['Centro'].isin(new_points)]
agro14_points = pd.concat([agro13_points, df_new]).reset_index(drop=True)
agro14_points

In [ ]:
df5 = files[4]
agro5 = df5[df5['DESCRIPCIO'].isin([c for c in df5['DESCRIPCIO'].unique() if c.startswith("AGR")])]
agro5['Centro'] = agro5.geometry.centroid
agro5_points = agro5[['OBJECTID', 'Centro']].reset_index(drop=True)
agro5_points


In [ ]:
pass4 = add_points(agro14_points, agro5_points)
new_points = pass4[pass4 > 9_000].dropna(axis=1, how="any").columns

In [ ]:
df_new = agro5_points[agro5_points['Centro'].isin(new_points)]
agro15_points = pd.concat([agro14_points, df_new]).reset_index(drop=True)
agro15_points

In [ ]:
df6 = files[5]

agro6 = df6[df6['TIP_INFO']=="AGRÍCOLA-PECUARIA-FORESTAL"]
agro6['Centro'] = agro6.geometry.centroid
agro6_points = agro6[['CLAVE', 'Centro']].reset_index(drop=True)
agro6_points

In [ ]:
agro15_points.drop(columns="geometry").to_file("Test15.shp")

In [ ]:
pass6 = add_points(agro15_points, agro6_points)
new_points = pass6[pass6 > 9_000].dropna(axis=1, how="any").columns

In [ ]:
df_new = agro6_points[agro6_points['Centro'].isin(new_points)]
agro16_points = pd.concat([agro15_points, df_new]).reset_index(drop=True)
agro16_points

In [ ]:
df7 = files[6]
agro7 = df7[df7['DESCRIPCIO'].isin([c for c in df7['DESCRIPCIO'].unique() if c.startswith("AGR")])]
agro7['Centro'] = agro7.geometry.centroid
agro7_points = agro7[['CLAVE', 'Centro']].reset_index(drop=True)
agro7_points

In [ ]:
pass7 = add_points(agro16_points, agro7_points)
new_points = pass7[pass7 > 9_000].dropna(axis=1, how="any").columns


In [ ]:
agro16_points = agro16_points.to_crs(epsg=6372)


In [ ]:
df_new= df_new.set_geometry("Centro")
df_new = df_new.to_crs(epsg=6372)

In [ ]:
df_new = agro7_points[agro7_points['Centro'].isin(new_points)]
df_new= df_new.set_geometry("Centro")
agro16_points = agro16_points.set_geometry("Centro")
df_new = df_new.to_crs(epsg=6372)
agro16_points = agro16_points.to_crs(epsg=6372)
agro1
agro17_points = pd.concat([agro16_points, df_new]).reset_index(drop=True)
agro17_points

In [ ]:
agro17_points = agro17_points.drop(columns="geometry")
agro17_points = agro17_points.set_geometry("Centro")
agro17_points.to_file("Final.shp")

# Archivos complementarios

In [ ]:
from config.paths import datapaths
from config.links import USOV_API
import geopandas as gpd

comp_3 = datapaths['raw']['INEGI'] / "702825007022_s"/"shp"/"us250s3c"/"us250s3c.shp"
comp_4 = datapaths['raw']['INEGI'] / "702825007023_s"/"shape"/"us250s4c"/"us250s4c.shp"
comp_5 = datapaths['raw']['INEGI'] / "702825007024_s" / "conjunto_de_datos"/"CAPAS"/"usv250s5c"/"usv250s5c.shp"
comp_6 = datapaths['raw']['INEGI'] / "889463598459_s"/"conjunto_de_datos"/ "usv250s6c.shp"
comp_files = [gpd.read_file(f) for f in [comp_3, comp_4, comp_5, comp_6]]

In [17]:
df1 = comp_files[0]
agro1c = df1[df1['TIP_INFO'] == "AGRICOLA-PECUARIA-FORESTAL"]

df2 = comp_files[1]
agro2c = df2[df2['TIP_INFO'] == "AGRICOLA-PECUARIA-FORESTAL"]

df3 = comp_files[2]
agro3c = df3[df3['Tip_info'] == "Agrícola-pecuaria-forestal"]

df4 = comp_files[3]
agro4c = df4[df4['Tip_info'] == "Agrícola-pecuaria-forestal"]


,OBJECTID,Tip_info,Ptoverif,Tip_pto,Culti1,Culti2,Culti3,Clave_conj,Coord_x,Coord_y,Cal_pos,geometry
0,1,Agricola-pecuaria-forestal,4,Verificació,65.9,120.2,120.3,F1404,2539830,1113322,Definida,POINT (2539829.785 1113321.513)
1,2,Agricola-pecuaria-forestal,3,Verificació,65.9,0,0,F1404,2543220,1117210,Definida,POINT (2543219.789 1117210.023)
2,3,Agricola-pecuaria-forestal,N,Observación,120,0,0,F1404,2553521,1124742,Definida,POINT (2553520.556 1124742.275)
3,4,Agricola-pecuaria-forestal,N,Observación,65,0,0,F1404,2549425,1119845,Definida,POINT (2549424.797 1119844.828)
4,5,Agricola-pecuaria-forestal,N,Observación,98,0,0,F1404,2543154,1118638,Definida,POINT (2543154.16 1118638.326)
...,...,...,...,...,...,...,...,...,...,...,...,...
948,1135,Agricola-pecuaria-forestal,N,Observación,162,0,0,H1208,1556514,1932808,Definida,POINT (1556514 1932808)
949,1136,Agricola-pecuaria-forestal,N,Observación,45,28,140,H1208,1552963,1920743,Definida,POINT (1552963.113 1920743.4)
950,1137,Agricola-pecuaria-forestal,5,Verificació,125,98,65,H1209,1895917,1916481,Definida,POINT (1895916.74 1916481.338)
951,1138,Agricola-pecuaria-forestal,6,Verificació,102,0,0,H1209,1904801,1910540,Definida,POINT (1904801.327 1910540.208)


In [21]:
import pandas as pd

agro_comp = pd.concat([agro1c, agro2c, agro3c, agro4c])
agro_comp_points = agro_comp[['CODIGO', "geometry"]]
agro_comp_points

,CODIGO,geometry
0,10000000051,POINT (3457852.255 555553.489)
1,10000000051,POINT (3422235.18 557031.632)
2,10000000051,POINT (3379842.101 543806.167)
3,10000000050,POINT (3465176.631 515909.346)
4,10000000051,POINT (3477428.685 556478.551)
...,...,...
1224,NaN,POINT (2544358.75 1113983.25)
1225,NaN,POINT (2509067.5 1113285.125)
1226,NaN,POINT (2685515.75 1112831.75)
1227,NaN,POINT (2662215.75 1112457.625)


In [22]:
final_data = gpd.read_file("/home/dusty/Documents/Repositories/proyecto_agronomia/notebooks/Final.shp")
final_data

,CODIGO,OBJECTID,CLAVE,geometry
0,10101040104,NaN,NaN,POINT (1135724.956 2347096.596)
1,10101040104,NaN,NaN,POINT (1138560.439 2348345.516)
2,10102040102,NaN,NaN,POINT (1256224.034 2325192.857)
3,10102040103,NaN,NaN,POINT (1133252.195 2346614.737)
4,10101040104,NaN,NaN,POINT (1108040.623 2347662.982)
...,...,...,...,...
26111,NaN,NaN,TP,POINT (3626269.02 804201.389)
26112,NaN,NaN,TP,POINT (3829446.525 1060001.431)
26113,NaN,NaN,TS,POINT (3381378.478 527989.195)
26114,NaN,NaN,TS,POINT (3125721.004 670571.591)


In [25]:
def add_points(base, new):
    distances = pd.DataFrame(index=base['geometry'], columns=new['geometry'])
    dims = distances.shape
    t = 0
    for p in distances.columns:
        if t % 1000==0:
            print(t / dims[1] * 100)
        distances[p] = base['geometry'].apply(lambda x: x.distance(p)).tolist()
        t+=1
    return distances[distances > 9_000]

last_update = add_points(final_data, agro_comp_points)

0.0
14.57938474996355
29.1587694999271
43.738154249890655
58.3175389998542
72.89692374981776
87.47630849978131


In [28]:
new_points = last_update[last_update > 9_000].dropna(axis=1, how="any").columns
new_points

Index([  POINT (3370894.699307091 513979.3379634477),
       POINT (3500624.4491360663 510944.42271747976),
          POINT (4011075.34033601 969614.0079282289),
         POINT (4014115.887595642 988121.1499636447),
        POINT (3957345.031199983 1005796.7621280797),
       POINT (4063547.3305634824 1077110.3411095599),
         POINT (3993285.713506188 765506.2428793773),
        POINT (3993369.7790954933 771602.3803998982),
        POINT (4051098.914459628 1089558.7284639135),
         POINT (3722063.012962406 964693.2400518461),
       ...
         POINT (2093779.773500003 2170073.959399998),
        POINT (2255084.8180000037 1855776.191399999),
                             POINT (2090508 2174864),
                       POINT (1990077.375 2009268.5),
                             POINT (2242641 1887637),
                        POINT (2200501.5 1661429.75),
       POINT (1982549.8916999996 1372345.4307999983),
       POINT (2148428.2536000013 1360747.6064000018),
        POINT (21